In [0]:
dbutils.library.restartPython()

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/raw_datasets/"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

FILE_PATH = "/Volumes/workspace/default/raw_datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df_raw = spark.read.csv(
    FILE_PATH,
    header      = True,
    inferSchema = True,
    nullValue   = " "    
)
 
print(f"\nRows    : {df_raw.count():,}")   
print(f"Columns : {len(df_raw.columns)}") 
print("\nSchema:")
df_raw.printSchema()

In [0]:
print("First 5 rows:")
display(df_raw.limit(5))

In [0]:
null_counts = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
])
 
print("Null counts per column:")
display(null_counts)

In [0]:
# TotalCharges is string because some rows are blank
# Cast to double and fill nulls with median
 
df = df_raw.withColumn(
    "TotalCharges",
    F.col("TotalCharges").cast(DoubleType())
)
 
median_val = df.approxQuantile("TotalCharges", [0.5], 0.01)[0]
df = df.fillna({"TotalCharges": median_val})
 
# Target column: Churn Yes/No → 1/0
df = df.withColumn(
    "label",
    F.when(F.col("Churn") == "Yes", 1).otherwise(0)
)
 
print(f"TotalCharges median fill value: {median_val:.2f}")
print(f"Nulls after fix: {df.filter(F.col('TotalCharges').isNull()).count()}")

In [0]:
print("\nChurn distribution:")
display(
    df.groupBy("Churn", "label")
      .count()
      .orderBy("label")
)

In [0]:
print("Summary statistics — numeric columns:")
display(
    df.select("tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen")
      .describe()
)

In [0]:
df.createOrReplaceTempView("churn_data")
 
print(f"\nFinal dataframe: {df.count():,} rows x {len(df.columns)} columns")
print("Temp view 'churn_data' created — you can now run SQL queries")
print("\nSample SQL query:")
display(spark.sql("""
    SELECT Contract,
           COUNT(*) AS customers,
           ROUND(AVG(label)*100, 1) AS churn_rate_pct
    FROM churn_data
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
"""))

In [0]:
df.write.mode("overwrite").format("delta").save(
    "/Volumes/workspace/default/raw_datasets/cleaned_data"
)